In [ ]:
#========================================================================
# Name: filter_wrf_dsd_properties_congestus_LASSO.ipynb
# Author: McKenna W. Stanford
# Author Contact: mckenna.stanford@pnnl.gov
#
# Description: Filter WRF DSD properties for congestus clouds and calculate
#              derived microphysical parameters for LASSO 2.5km domain.
#========================================================================

In [14]:
#===============================
# Imports
#===============================
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import glob
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
import datetime
import matplotlib.colors as colors
import matplotlib as mpl
from pytz import utc
from matplotlib.lines import Line2D
import pickle
from scipy.spatial import cKDTree
import os
import dask
from distributed import Client, LocalCluster
from dask import delayed, compute
import dask.array as da

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import cartopy.io.img_tiles as cimgt
import shapely.geometry as sgeom
import matplotlib.patheffects as path_effects

import warnings
warnings.filterwarnings("ignore")

plt.rcParams['text.usetex'] = True

In [15]:
# Your base directory
wrf_path = '/pscratch/sd/m/mckenna/wrf_lasso/coarse_grained/'

# d2
search_pattern = os.path.join(wrf_path, '**', 'd2', 'derived', 'LASSO_CACTI_2.5km_derived_*.nc')
d2_ct_files = sorted(glob.glob(search_pattern, recursive=True))
search_pattern = os.path.join(wrf_path, '**', 'd2', 'derived', 'LASSO_CACTI_2.5km_DSD_*.nc')
d2_dsd_files = sorted(glob.glob(search_pattern, recursive=True))
search_pattern = os.path.join(wrf_path, '**', 'd2', 'cld_met_files', '*_cld*.nc')
d2_cld_files = sorted(glob.glob(search_pattern, recursive=True))
search_pattern = os.path.join(wrf_path, '**', 'd2', 'cld_met_files', '*_met*.nc')
d2_met_files = sorted(glob.glob(search_pattern, recursive=True))
num_d2_ct_files = len(d2_ct_files)
num_d2_dsd_files = len(d2_dsd_files)
num_d2_cld_files = len(d2_cld_files)
num_d2_met_files = len(d2_met_files)
print(f'# of d2_ct files found: {num_d2_ct_files}')
print(f'# of d2_dsd files found: {num_d2_dsd_files}')
print(f'# of d2_cld files found: {num_d2_cld_files}')
print(f'# of d2_met files found: {num_d2_met_files}')

# d3
search_pattern = os.path.join(wrf_path, '**', 'd3', 'derived', 'LASSO_CACTI_2.5km_derived_*.nc')
d3_ct_files = sorted(glob.glob(search_pattern, recursive=True))
search_pattern = os.path.join(wrf_path, '**', 'd3', 'derived', 'LASSO_CACTI_2.5km_DSD_*.nc')
d3_dsd_files = sorted(glob.glob(search_pattern, recursive=True))
search_pattern = os.path.join(wrf_path, '**', 'd3', 'cld_met_files', '*_cld_*.nc')
d3_cld_files = sorted(glob.glob(search_pattern, recursive=True))
search_pattern = os.path.join(wrf_path, '**', 'd3', 'cld_met_files', '*_met_*.nc')
d3_met_files = sorted(glob.glob(search_pattern, recursive=True))
num_d3_ct_files = len(d3_ct_files)
num_d3_dsd_files = len(d3_dsd_files)
num_d3_cld_files = len(d3_cld_files)
num_d3_met_files = len(d3_met_files)
print(f'# of d3_ct files found: {num_d3_ct_files}')
print(f'# of d3_dsd files found: {num_d3_dsd_files}')
print(f'# of d3_cld files found: {num_d3_cld_files}')
print(f'# of d3_met files found: {num_d3_met_files}')

# d4
search_pattern = os.path.join(wrf_path, '**', 'd4', 'derived', 'LASSO_CACTI_2.5km_derived_*.nc')
d4_ct_files = sorted(glob.glob(search_pattern, recursive=True))
search_pattern = os.path.join(wrf_path, '**', 'd4', 'derived', 'LASSO_CACTI_2.5km_DSD_*.nc')
d4_dsd_files = sorted(glob.glob(search_pattern, recursive=True))
search_pattern = os.path.join(wrf_path, '**', 'd4', 'cld_met_files', '*_cld_*.nc')
d4_cld_files = sorted(glob.glob(search_pattern, recursive=True))
search_pattern = os.path.join(wrf_path, '**', 'd4', 'cld_met_files', '*_met_*.nc')
d4_met_files = sorted(glob.glob(search_pattern, recursive=True))
num_d4_ct_files = len(d4_ct_files)
num_d4_dsd_files = len(d4_dsd_files)
num_d4_cld_files = len(d4_cld_files)
num_d4_met_files = len(d4_met_files)
print(f'# of d4_ct files found: {num_d4_ct_files}')
print(f'# of d4_dsd files found: {num_d4_dsd_files}')
print(f'# of d4_cld files found: {num_d4_cld_files}')
print(f'# of d4_met files found: {num_d4_met_files}')

# of d2_ct files found: 392
# of d2_dsd files found: 392
# of d2_cld files found: 392
# of d2_met files found: 392
# of d3_ct files found: 384
# of d3_dsd files found: 384
# of d3_cld files found: 384
# of d3_met files found: 384
# of d4_ct files found: 392
# of d4_dsd files found: 392
# of d4_cld files found: 392
# of d4_met files found: 392


In [16]:
csapr_path = '/pscratch/sd/m/mckenna/cacti/csapr_regridded/'
csapr_files = sorted(glob.glob(csapr_path+'/*.nc'))
#ds = xr.open_dataset(csapr_files[0])

In [17]:
def get_csapr_mask(csapr_file, cld_file):
    """
    Creates a rectangular mask for the CLD grid based on the CSAPR domain.
    The final mask is a perfect rectangle in grid-index space.
    """
    with xr.open_dataset(csapr_file) as ds_csapr:
        lon_csapr = ds_csapr['point_longitude'].squeeze()
        lat_csapr = ds_csapr['point_latitude'].squeeze()

    with xr.open_dataset(cld_file) as ds_cld:
        lon_cld = ds_cld['XLONG'].squeeze()
        lat_cld = ds_cld['XLAT'].squeeze()

    # --- Step 1: Create the initial geographic mask (as before) ---
    lat_min_csapr = lat_csapr.min().item()
    lat_max_csapr = lat_csapr.max().item()
    lon_min_csapr = lon_csapr.min().item()
    lon_max_csapr = lon_csapr.max().item()

    geographic_mask = (
        (lon_cld >= lon_min_csapr) & (lon_cld <= lon_max_csapr) &
        (lat_cld >= lat_min_csapr) & (lat_cld <= lat_max_csapr)
    ).values  # Work with numpy array from here

    # --- Step 2: Find the min/max row and column indices of the mask ---
    # .any(axis=0) -> checks which COLUMNS have at least one 'True'
    # .any(axis=1) -> checks which ROWS have at least one 'True'
    
    # Find the first and last column index that contain a valid point
    cols_with_data = np.where(geographic_mask.any(axis=0))[0]
    if len(cols_with_data) == 0: # Handle case where there's no overlap
        return np.zeros_like(lon_cld.values, dtype=float)
    j_min, j_max = cols_with_data.min(), cols_with_data.max()

    # Find the first and last row index that contain a valid point
    rows_with_data = np.where(geographic_mask.any(axis=1))[0]
    i_min, i_max = rows_with_data.min(), rows_with_data.max()

    # --- Step 3 & 4: Create a new mask and fill the rectangular region ---
    rectangular_mask = np.zeros_like(geographic_mask, dtype=float)
    # Use slicing to fill the bounding box in index space
    rectangular_mask[i_min : i_max + 1, j_min : j_max + 1] = 1.0
    
    # Return the final rectangular mask and the corner indices for visualization
    corner_indices = {'i_min': i_min, 'i_max': i_max, 'j_min': j_min, 'j_max': j_max}
    #return rectangular_mask, lon_cld.values, lat_cld.values, corner_indices
    return rectangular_mask

In [18]:
def subset_var_3d(data,mask):

    nz = len(data[:,0,0])
    # mask values where the csapr mask == 0.
    reshaped_data_3d = []
    
    for kk in range(nz):
        data_2d = data[kk,:,:]
        masked_data = np.ma.masked_array(data_2d, mask=mask == 0)
        #print(aaaaa)
        
        # Extract the 2D subset defined by the mask
        # The unmasked elements will define the dimensions
        unmasked_data = masked_data[~masked_data.mask]  # Extract non-masked data
        
        
        row_indices, col_indices = np.where(~masked_data.mask)  # Get the unmasked indices
        
        new_rows = len(np.unique(row_indices))  # Number of unique rows in the unmasked region
        new_cols = len(np.unique(col_indices))  # Number of unique columns in the unmasked region
        
        reshaped_data = unmasked_data.reshape((new_rows, new_cols))
    
        reshaped_data_3d.append(reshaped_data.data)

    reshaped_data_3d = np.asarray(reshaped_data_3d)
    
    return reshaped_data_3d

In [19]:
def subset_var(data,mask):

    # mask values where the csapr mask == 0.
    masked_data = np.ma.masked_array(data, mask=mask == 0)

    # Extract the 2D subset defined by the mask
    # The unmasked elements will define the dimensions
    unmasked_data = masked_data[~masked_data.mask]  # Extract non-masked data


    row_indices, col_indices = np.where(~masked_data.mask)  # Get the unmasked indices

    new_rows = len(np.unique(row_indices))  # Number of unique rows in the unmasked region
    new_cols = len(np.unique(col_indices))  # Number of unique columns in the unmasked region

    reshaped_data = unmasked_data.reshape((new_rows, new_cols))

    return reshaped_data.data

In [20]:
def trim_var(lon,lat,var,var_dim):
    lonmin = -64.95
    lonmax = -64.65
    latmin = -32.9
    latmax = -31.1
    
    mean_lon = np.mean(lon,axis=0)
    mean_lat = np.mean(lat,axis=1)  
    
    # Longitude
    trim_id = np.where( (mean_lon >= lonmin) & (mean_lon <= lonmax)  )[0]
    if var_dim == 2:
        var = var[:,trim_id]
    elif var_dim == 3:
        var = var[:,:,trim_id]

    # Latitude
    trim_id = np.where( (mean_lat >= latmin) & (mean_lat <= latmax)  )[0]
    if var_dim == 2:
        var = var[trim_id,:]
    elif var_dim == 3:
        var = var[:,trim_id,:]
        
    return var

In [21]:
for ii in range(0,1):
#def driver_func(ct_file,dsd_file,cld_file):

    ct_file = d2_ct_files[ii]
    dsd_file = d2_dsd_files[ii]
    cld_file = d2_cld_files[ii]
    met_file = d2_met_files[ii]
    print('CT file:',ct_file)
    print('DSD file:',dsd_file)
    print('cld file:',cld_file)
    print('met file:',met_file)

    ct_str = ct_file.split('/')[-1].split('_')
    ct_date_str = ct_str[4]
    ct_time_str = ct_str[5].split(':')
    ct_datetime = datetime.datetime(int(ct_date_str[0:4]),int(ct_date_str[4:6]),int(ct_date_str[6:8]),int(ct_time_str[0]),int(ct_time_str[1]),int(ct_time_str[2]))
    print(ct_datetime)

    dsd_str = dsd_file.split('/')[-1].split('_')
    dsd_date_str = dsd_str[7]
    dsd_time_str = dsd_str[8].split(':')
    dsd_datetime = datetime.datetime(int(dsd_date_str[0:4]),int(dsd_date_str[4:6]),int(dsd_date_str[6:8]),int(dsd_time_str[0]),int(dsd_time_str[1]),int(dsd_time_str[2]))
    print(dsd_datetime)

    cld_str = cld_file.split('/')[-1].split('_')[-1].split('.')
    cld_date_str = cld_str[2]
    cld_time_str = cld_str[3]
    cld_datetime = datetime.datetime(int(cld_date_str[0:4]),int(cld_date_str[4:6]),int(cld_date_str[6:8]),int(cld_time_str[0:2]),int(cld_time_str[2:4]),int(cld_time_str[4:6]))
    print(cld_datetime)
    
    met_str = met_file.split('/')[-1].split('_')[-1].split('.')
    met_date_str = met_str[2]
    met_time_str = met_str[3]
    met_datetime = datetime.datetime(int(met_date_str[0:4]),int(met_date_str[4:6]),int(met_date_str[6:8]),int(met_time_str[0:2]),int(met_time_str[2:4]),int(met_time_str[4:6]))
    print(met_datetime)
    
    print(aaaa)

CT file: /pscratch/sd/m/mckenna/wrf_lasso/coarse_grained/20181129/gefs03/d2/derived/LASSO_CACTI_2.5km_derived_20181129_12:00:00_gefs03_d2_native.nc
DSD file: /pscratch/sd/m/mckenna/wrf_lasso/coarse_grained/20181129/gefs03/d2/derived/LASSO_CACTI_2.5km_DSD_parameters_liq_only_20181129_12:00:00_gefs03_d2_native.nc
cld file: /pscratch/sd/m/mckenna/wrf_lasso/coarse_grained/20181129/gefs03/d2/cld_met_files/corlasso_cld_2018112900gefs03d2_base_M1.m1.20181129.120000.nc
met file: /pscratch/sd/m/mckenna/wrf_lasso/coarse_grained/20181129/gefs03/d2/cld_met_files/corlasso_met_2018112900gefs03d2_base_M1.m1.20181129.120000.nc
2018-11-29 12:00:00
2018-11-29 12:00:00
2018-11-29 12:00:00
2018-11-29 12:00:00


NameError: name 'aaaa' is not defined

In [22]:
#for ii in range(150,151):
def driver_func(ct_file,dsd_file,cld_file,met_file,csapr_file,domain):
    
    if False:
        csapr_file = csapr_files[0]
        domain='d2'
        ct_file = d2_ct_files[ii]
        dsd_file = d2_dsd_files[ii]
        cld_file = d2_cld_files[ii]
        met_file = d2_met_files[ii]
        print('CT file:',ct_file)
        print('DSD file:',dsd_file)
        print('cld file:',cld_file)
        print('met file:',met_file)

    ct_str = ct_file.split('/')[-1].split('_')
    ct_date_str = ct_str[4]
    ct_time_str = ct_str[5].split(':')
    ct_datetime = datetime.datetime(int(ct_date_str[0:4]),int(ct_date_str[4:6]),int(ct_date_str[6:8]),int(ct_time_str[0]),int(ct_time_str[1]),int(ct_time_str[2]))
    print(ct_datetime)

    dsd_str = dsd_file.split('/')[-1].split('_')
    dsd_date_str = dsd_str[7]
    dsd_time_str = dsd_str[8].split(':')
    dsd_datetime = datetime.datetime(int(dsd_date_str[0:4]),int(dsd_date_str[4:6]),int(dsd_date_str[6:8]),int(dsd_time_str[0]),int(dsd_time_str[1]),int(dsd_time_str[2]))
    print(dsd_datetime)

    cld_str = cld_file.split('/')[-1].split('_')[-1].split('.')
    cld_date_str = cld_str[2]
    cld_time_str = cld_str[3]
    cld_datetime = datetime.datetime(int(cld_date_str[0:4]),int(cld_date_str[4:6]),int(cld_date_str[6:8]),int(cld_time_str[0:2]),int(cld_time_str[2:4]),int(cld_time_str[4:6]))
    print(cld_datetime)
    
    met_str = met_file.split('/')[-1].split('_')[-1].split('.')
    met_date_str = met_str[2]
    met_time_str = met_str[3]
    met_datetime = datetime.datetime(int(met_date_str[0:4]),int(met_date_str[4:6]),int(met_date_str[6:8]),int(met_time_str[0:2]),int(met_time_str[2:4]),int(met_time_str[4:6]))
    print(met_datetime)

    
    if ct_datetime != dsd_datetime:
        raise RuntimeError('Times between cloud-top file and DSD file do not match...')

    if ct_datetime != met_datetime:
        raise RuntimeError('Times between cloud-top file and MET file do not match...')
    
    ds_ct = xr.open_dataset(ct_file)
    cth = ds_ct['cth_tau_ir'].values 
    #==============================================================================
    # Skip file entirely if entirely absent of cloud
    #==============================================================================
    if np.isnan(np.nanmax(cth)):
        ds_ct.close()
        print('Skipping file because no cloud present.')
        #return
    ctt = ds_ct['ctt_tau_ir'] 
    #==============================================================================
    # Skip file if the warmest CTT in the entire domain is colder than -20 deg C.
    #==============================================================================
    if np.nanmax(ctt) < -20.:
        ds_ct.close()
        print('Skipping file because warmest CTT is colder than -20 deg C.')
        #return
        
    lon = ds_ct['lon']
    lat = ds_ct['lat']
    
    ctt_dim = 2
    ctt_trim = trim_var(lon,lat,ctt,2)
    lon_trim = trim_var(lon,lat,lon,2)
    lat_trim = trim_var(lon,lat,lat,2)

    #==============================================================================
    # Skip file if warmest CTT in the trimmed domain is colder than -20 deg C.
    #==============================================================================
    if (np.nanmax(ctt_trim) < -20.) or (np.isnan(np.nanmax(ctt_trim))):
        ds_ct.close()
        print('Skipping file because warmest CTT in trimmed domain is colder than -20 deg C or is NaN.')
        #return
        
    # Read in other variables to be used for filtering
    lwp = ds_ct['lwp']
    twp = ds_ct['twp']
    iwp = ds_ct['iwp']
    opd = ds_ct['opd_vis']

    # Trim them
    lwp_trim = trim_var(lon,lat,lwp,2)
    twp_trim = trim_var(lon,lat,twp,2)
    iwp_trim = trim_var(lon,lat,iwp,2)
    opd_trim = trim_var(lon,lat,opd,2)
    cth_trim = trim_var(lon,lat,cth,2)
    ds_ct.close()

    #==============================================================================
    #==============================================================================
    # Now extract DSD parameters
    #==============================================================================
    #==============================================================================
    ds_dsd = xr.open_dataset(dsd_file)
    temp = ds_dsd['temp'] # deg C
    zamsl = ds_dsd['zamsl'] # m
    D_num = ds_dsd['D_num'] # m
    D_vol = ds_dsd['D_vol'] # m
    D_mass = ds_dsd['D_mass'] # m
    mmd = ds_dsd['mmd'] # m
    N = ds_dsd['N'] # kg^-1
    q = ds_dsd['q'] # kg/kg
    rho_air = ds_dsd['rho_air'] # kg/m^3
    N_0_r = ds_dsd['N_0_r'] # m^-4
    N_0_c = ds_dsd['N_0_c'] # m^-4
    lambda_r = ds_dsd['lambda_r'] # m^-1
    lambda_c = ds_dsd['lambda_c'] # m^-1
    mu_c = ds_dsd['mu_c'] # unitless
    q_gt_100um = ds_dsd['q_gt_100um'] # kg/kg
    N_gt_100um = ds_dsd['N_gt_100um'] # kg^-1
    ds_dsd.close()


    zamsl_trim = trim_var(lon,lat,zamsl,3)
    temp_trim = trim_var(lon,lat,temp,3)
    D_num_trim = trim_var(lon,lat,D_num,3)
    D_vol_trim = trim_var(lon,lat,D_vol,3)
    D_mass_trim = trim_var(lon,lat,D_mass,3)
    mmd_trim = trim_var(lon,lat,mmd,3)
    N_trim = trim_var(lon,lat,N,3)
    q_trim = trim_var(lon,lat,q,3)
    rho_air_trim = trim_var(lon,lat,rho_air,3)
    N_0_r_trim = trim_var(lon,lat,N_0_r,3)
    N_0_c_trim = trim_var(lon,lat,N_0_c,3)
    lambda_r_trim = trim_var(lon,lat,lambda_r,3)
    lambda_c_trim = trim_var(lon,lat,lambda_c,3)
    mu_c_trim = trim_var(lon,lat,mu_c,3)
    q_gt_100um_trim = trim_var(lon,lat,q_gt_100um,3)
    N_gt_100um_trim = trim_var(lon,lat,N_gt_100um,3)

    #------------------------------------------
    # Get temperature
    # Also get Qr, Qc. Don't actually need these,
    # but they're helpful conceptually during testing.
    #------------------------------------------

    if cld_datetime != ct_datetime:
        raise RuntimeError('Times between cloud-top file and cld file do not match...')

        
    ds_3d = xr.open_dataset(cld_file)
    qr = ds_3d['QRAIN'].squeeze() # kg/kg
    qc = ds_3d['QCLOUD'].squeeze() # kg/kg
    Nr = ds_3d['QNRAIN'].squeeze() # kg/kg
    Nc = ds_3d['QNCLOUD'].squeeze() # kg/kg
    lon_3d = ds_3d['XLONG'].squeeze()
    lat_3d = ds_3d['XLAT'].squeeze()
    ds_3d.close()

    csapr_mask = get_csapr_mask(csapr_file,cld_file)
    qr = subset_var_3d(qr,csapr_mask)
    qc = subset_var_3d(qc,csapr_mask)
    Nr = subset_var_3d(Nr,csapr_mask)
    Nc = subset_var_3d(Nc,csapr_mask)
    lon_3d = subset_var(lon_3d,csapr_mask)
    lat_3d = subset_var(lat_3d,csapr_mask)

    
    qr_trim = trim_var(lon_3d,lat_3d,qr,3)
    Nr_trim = trim_var(lon_3d,lat_3d,Nr,3)
    qc_trim = trim_var(lon_3d,lat_3d,qc,3)
    Nc_trim = trim_var(lon_3d,lat_3d,Nc,3)

    if True:
        ds_3d_met = xr.open_dataset(met_file)
        qv = ds_3d_met['QVAPOR'].squeeze() # kg/kg
        tmp_temp = ds_3d_met['TEMPERATURE'].squeeze() # K
        pres = ds_3d_met['PRESSURE'].squeeze()*1.e2 # Pa
        lon_3d = ds_3d_met['XLONG'].squeeze()
        lat_3d = ds_3d_met['XLAT'].squeeze()
        ds_3d_met.close()

        pres = subset_var_3d(pres,csapr_mask)
        tmp_temp = subset_var_3d(tmp_temp,csapr_mask)
        qv = subset_var_3d(qv,csapr_mask)

        lon_3d = subset_var(lon_3d,csapr_mask)
        lat_3d = subset_var(lat_3d,csapr_mask)
        
        # Calculate RH
        C0 = 0.611583699e3
        C1 = 0.444606896e2
        C2 = 0.143177157e1
        C3 = 0.264224321e-1
        C4 = 0.299291081e-3
        C5 = 0.203154182e-5
        C6 = 0.702620698e-8
        C7 = 0.379534310e-11
        C8 = -0.321582393e-13
        X = tmp_temp - 273.15 # temperature in deg C
        ESL = C0 + X*(C1 + X*(C2 + X*(C3 + X*(C4 + X*(C5 + X*(C6 + X*(C7 + X*C8))))))) #saturation vapor pressure
        QVS = 0.622*ESL/(pres - ESL) #saturation vapor mixing ratio
        RH = 1e2*qv/QVS # %
        #RH_trim = trim_var(lon_3d,lat_3d,RH,3)

    ds_3d = xr.open_dataset(met_file)
    wa = ds_3d['WA'].squeeze() # m/s
    RH_native = ds_3d['RH'].squeeze() # %
    lon_3d = ds_3d['XLONG'].squeeze()
    lat_3d = ds_3d['XLAT'].squeeze()
    ds_3d.close()

    wa = subset_var_3d(wa,csapr_mask)
    RH_native = subset_var_3d(RH_native,csapr_mask)
    lon_3d = subset_var(lon_3d,csapr_mask)
    lat_3d = subset_var(lat_3d,csapr_mask)
    
    wa_trim = trim_var(lon_3d,lat_3d,wa,3)
    RH_trim = trim_var(lon_3d,lat_3d,RH,3)
    RH_native_trim = trim_var(lon_3d,lat_3d,RH_native,3)

    #------------------------------------------
    # Limit profiles to be under 10 km, as a start to limit memory
    #------------------------------------------
    zamsl_mean = np.mean(zamsl_trim,axis=(1,2))
    temp_mean = np.mean(temp_trim,axis=(1,2))
    height_id = np.where(zamsl_mean < 10000.)[0]

    temp_trim = temp_trim[height_id,:,:]
    RH_trim = RH_trim[height_id,:,:]
    RH_native_trim = RH_native_trim[height_id,:,:]
    qr_trim = qr_trim[height_id,:,:]
    Nr_trim = Nr_trim[height_id,:,:]
    qc_trim = qc_trim[height_id,:,:]
    Nc_trim = Nc_trim[height_id,:,:]
    wa_trim = wa_trim[height_id,:,:]
    zamsl_trim = zamsl_trim[height_id,:,:]
    D_num_trim = D_num_trim[height_id,:,:]
    D_vol_trim = D_vol_trim[height_id,:,:]
    D_mass_trim = D_mass_trim[height_id,:,:]
    mmd_trim = mmd_trim[height_id,:,:]
    N_trim = N_trim[height_id,:,:]
    q_trim = q_trim[height_id,:,:]
    rho_air_trim = rho_air_trim[height_id,:,:]
    N_0_r_trim = N_0_r_trim[height_id,:,:]
    N_0_c_trim = N_0_c_trim[height_id,:,:]
    lambda_r_trim = lambda_r_trim[height_id,:,:]
    lambda_c_trim = lambda_c_trim[height_id,:,:]
    mu_c_trim = mu_c_trim[height_id,:,:]
    q_gt_100um_trim = q_gt_100um_trim[height_id,:,:]
    N_gt_100um_trim = N_gt_100um_trim[height_id,:,:]


    #------------------------------------------
    # Identify profiles where CTT > -20 deg C to isolate
    # cumulus and congestus profiles. Resulting arrays will
    # be of shape nz x num_cong_profiles
    #------------------------------------------ 

    nz_trim = len(zamsl_trim[:,0,0])
    cong_mask = (ctt_trim >= -20.) & (cth_trim >= 0.5) & (opd_trim > 10.)

    flat_mask = cong_mask.values.flatten()
    zamsl_flat = zamsl_trim.values.reshape(nz_trim, -1)  
    zamsl_filtered = zamsl_flat[:, flat_mask]  
    temp_flat = temp_trim.values.reshape(nz_trim, -1)
    temp_filtered = temp_flat[:, flat_mask]  
    RH_flat = RH_trim.reshape(nz_trim, -1)
    RH_filtered = RH_flat[:, flat_mask]
    RH_native_flat = RH_native_trim.reshape(nz_trim, -1)
    RH_native_filtered = RH_native_flat[:, flat_mask]
    qr_flat = qr_trim.reshape(nz_trim, -1)
    qr_filtered = qr_flat[:, flat_mask]
    qc_flat = qc_trim.reshape(nz_trim, -1)
    qc_filtered = qc_flat[:, flat_mask]
    Nr_flat = Nr_trim.reshape(nz_trim, -1)
    Nr_filtered = Nr_flat[:, flat_mask]
    Nc_flat = Nc_trim.reshape(nz_trim, -1)
    Nc_filtered = Nc_flat[:, flat_mask]
    wa_flat = wa_trim.reshape(nz_trim, -1)
    wa_filtered = wa_flat[:, flat_mask]
    
    D_num_flat = D_num_trim.values.reshape(nz_trim, -1)
    D_num_filtered = D_num_flat[:, flat_mask]
    D_vol_flat = D_vol_trim.values.reshape(nz_trim, -1)
    D_vol_filtered = D_vol_flat[:, flat_mask]
    D_mass_flat = D_mass_trim.values.reshape(nz_trim, -1)
    D_mass_filtered = D_mass_flat[:, flat_mask]
    mmd_flat = mmd_trim.values.reshape(nz_trim, -1)
    mmd_filtered = mmd_flat[:, flat_mask]
    N_flat = N_trim.values.reshape(nz_trim, -1)
    N_filtered = N_flat[:, flat_mask]
    q_flat = q_trim.values.reshape(nz_trim, -1)
    q_filtered = q_flat[:, flat_mask]
    rho_air_flat = rho_air_trim.values.reshape(nz_trim, -1)
    rho_air_filtered = rho_air_flat[:, flat_mask]
    N_0_r_flat = N_0_r_trim.values.reshape(nz_trim, -1)
    N_0_r_filtered = N_0_r_flat[:, flat_mask]
    N_0_c_flat = N_0_c_trim.values.reshape(nz_trim, -1)
    N_0_c_filtered = N_0_c_flat[:, flat_mask]
    lambda_r_flat = lambda_r_trim.values.reshape(nz_trim, -1)
    lambda_r_filtered = lambda_r_flat[:, flat_mask]
    lambda_c_flat = lambda_c_trim.values.reshape(nz_trim, -1)
    lambda_c_filtered = lambda_c_flat[:, flat_mask]
    mu_c_flat = mu_c_trim.values.reshape(nz_trim, -1)
    mu_c_filtered = mu_c_flat[:, flat_mask]
    q_gt_100um_flat = q_gt_100um_trim.values.reshape(nz_trim, -1)
    q_gt_100um_filtered = q_gt_100um_flat[:, flat_mask]
    N_gt_100um_flat = N_gt_100um_trim.values.reshape(nz_trim, -1)
    N_gt_100um_filtered = N_gt_100um_flat[:, flat_mask]

    ctt_flat = ctt_trim.values.flatten()
    ctt_filtered = ctt_flat[flat_mask] 
    cth_flat = cth_trim.flatten()
    cth_filtered = cth_flat[flat_mask]  
    opd_flat = opd_trim.values.flatten()
    opd_filtered = opd_flat[flat_mask]  
    lwp_flat = lwp_trim.values.flatten()
    lwp_filtered = lwp_flat[flat_mask]  
    twp_flat = twp_trim.values.flatten()
    twp_filtered = twp_flat[flat_mask] 
    iwp_flat = iwp_trim.values.flatten()
    iwp_filtered = iwp_flat[flat_mask] 

    #---------------------------------------------------
    # Now limit to points w/ total LWC > 1.e-3 g/m^3
    #---------------------------------------------------
    lwc_filtered = q_filtered*rho_air_filtered*1.e3 # g/m^3
    valid_mask = (lwc_filtered > 1.e-3) & (temp_filtered > 0.)
    if np.size(valid_mask) == 0.:
        print('No congestus ID points w/ LWC > 1.e-3 g/m^3.')
        #return

    lwc_valid = lwc_filtered[valid_mask]
    temp_valid = temp_filtered[valid_mask]
    RH_valid = RH_filtered[valid_mask]
    RH_native_valid = RH_native_filtered[valid_mask]
    D_num_valid = D_num_filtered[valid_mask]
    D_vol_valid = D_vol_filtered[valid_mask]
    D_mass_valid = D_mass_filtered[valid_mask]
    mmd_valid = mmd_filtered[valid_mask]
    N_valid = N_filtered[valid_mask]
    q_valid = q_filtered[valid_mask]
    rho_air_valid = rho_air_filtered[valid_mask]
    N_0_r_valid = N_0_r_filtered[valid_mask]
    N_0_c_valid = N_0_c_filtered[valid_mask]
    lambda_r_valid = lambda_r_filtered[valid_mask]
    lambda_c_valid = lambda_c_filtered[valid_mask]
    mu_c_valid = mu_c_filtered[valid_mask]
    q_gt_100um_valid = q_gt_100um_filtered[valid_mask]
    N_gt_100um_valid = N_gt_100um_filtered[valid_mask]
    qr_valid = qr_filtered[valid_mask]
    Nr_valid = Nr_filtered[valid_mask]
    qc_valid = qc_filtered[valid_mask]
    Nc_valid = Nc_filtered[valid_mask]
    wa_valid = wa_filtered[valid_mask]
    zamsl_valid = zamsl_filtered[valid_mask]
    #print(np.shape(RH_valid),np.max(RH_valid),np.min(RH_valid))

    save_path = '/pscratch/sd/m/mckenna/cacti/wrf/temp_cong_dsd_params/'
    str_dt = dsd_datetime.strftime('%Y%m%d_%H:%M:%S')
    outname = save_path+f"LASSO_{domain}_cong_dsd_params_"+str_dt+".npz"
    np.savez(outname,\
                zamsl=zamsl_valid,\
                temp=temp_valid,\
                RH=RH_valid,\
                RH_native=RH_native_valid,\
                lwc=lwc_valid,\
                D_num=D_num_valid,\
                D_vol=D_vol_valid,\
                D_mass=D_mass_valid,\
                mmd=mmd_valid,\
                N=N_valid,\
                q=q_valid,\
                rho_air=rho_air_valid,\
                N_0_r=N_0_r_valid,\
                N_0_c=N_0_c_valid,\
                lambda_r=lambda_r_valid,\
                lambda_c=lambda_c_valid,\
                mu_c=mu_c_valid,\
                q_gt_100um=q_gt_100um_valid,\
                N_gt_100um=N_gt_100um_valid,\
                qr=qr_valid,\
                qc=qc_valid,\
                Nr=Nr_valid,\
                Nc=Nc_valid,\
                wa=wa_valid,\
                )
    
    return outname


In [ ]:
%%time
dumid = 143
domain='d2'
dum = driver_func(d2_ct_files[dumid],d2_dsd_files[dumid],d2_cld_files[dumid],d2_met_files[dumid],csapr_files[0],domain)
#domain='d3'
#dum = driver_func(d3_ct_files[dumid],d3_dsd_files[dumid],d3_cld_files[dumid],d3_met_files[dumid],domain)
#domain='d4'
#dum = driver_func(d4_ct_files[dumid],d4_dsd_files[dumid],d4_cld_files[dumid],d4_met_files[dumid],domain)

# Serial Run

In [16]:
domains = ['d2','d3','d4']
for jj in range(len(domains)):
    domain = domains[jj]
    print('--------------------------------------')
    print('DOMAIN:',domain)
    print('--------------------------------------')
    if domain == 'd2':
        tmp_ct_files = d2_ct_files
        tmp_dsd_files = d2_dsd_files
        tmp_cld_files = d2_cld_files
        tmp_met_files = d2_met_files
    elif domain == 'd3':
        tmp_ct_files = d3_ct_files
        tmp_dsd_files = d3_dsd_files
        tmp_cld_files = d3_cld_files
        tmp_met_files = d3_met_files
    elif domain == 'd4':
        tmp_ct_files = d4_ct_files
        tmp_dsd_files = d4_dsd_files
        tmp_cld_files = d4_cld_files
        tmp_met_files = d4_met_files

    start_id = 0
    end_id = len(tmp_ct_files)

    for ii in range(start_id,end_id):
        outname = driver_func(tmp_ct_files[ii],tmp_dsd_files[ii],tmp_cld_files[ii],tmp_met_files[ii],domain)


--------------------------------------
DOMAIN: d2
--------------------------------------
2018-11-29 12:00:00
2018-11-29 12:00:00
2018-11-29 12:00:00
2018-11-29 12:00:00
Skipping file because no cloud present.
2018-11-29 12:15:00
2018-11-29 12:15:00
2018-11-29 12:15:00
2018-11-29 12:15:00
Skipping file because no cloud present.
2018-11-29 12:30:00
2018-11-29 12:30:00
2018-11-29 12:30:00
2018-11-29 12:30:00
Skipping file because no cloud present.
2018-11-29 12:45:00
2018-11-29 12:45:00
2018-11-29 12:45:00
2018-11-29 12:45:00
Skipping file because no cloud present.
2018-11-29 13:00:00
2018-11-29 13:00:00
2018-11-29 13:00:00
2018-11-29 13:00:00
Skipping file because no cloud present.
2018-11-29 13:15:00
2018-11-29 13:15:00
2018-11-29 13:15:00
2018-11-29 13:15:00
2018-11-29 13:30:00
2018-11-29 13:30:00
2018-11-29 13:30:00
2018-11-29 13:30:00
2018-11-29 13:45:00
2018-11-29 13:45:00
2018-11-29 13:45:00
2018-11-29 13:45:00
2018-11-29 14:00:00
2018-11-29 14:00:00
2018-11-29 14:00:00
2018-11-29 

# Dask stuff

In [8]:
dask.config.config["distributed"]["dashboard"]["link"] = "{JUPYTERHUB_SERVICE_PREFIX}proxy/{host}:{port}/status" 

In [9]:
cluster = LocalCluster(n_workers=80,threads_per_worker=1,memory_limit='5GB')#,dashboard_address=':8787')
client = Client(cluster)

2026-06-27 12:14:45,638 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:34251' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'driver_func-370c73ae-34ca-4653-a978-9f9359562407', 'driver_func-d2eb2977-4207-4ee2-9fef-cb499ec2756c', 'driver_func-ff8e6fc1-4201-426c-a3ff-01fa3b1586c3', 'driver_func-9e05f482-71e5-4271-81e4-4ad1657e8c67', 'driver_func-67540997-4644-47cf-95d7-12dda6dc4aff'} (stimulus_id='handle-worker-cleanup-1782587685.6384094')
2026-06-27 12:14:45,670 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:43885' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'driver_func-4c5ae2d2-ff26-4e25-a34e-5bd4692503e6', 'driver_func-d7ef99f3-1671-4bcf-a1d5-d653a4f807f9', 'driver_func-a30fc84b-cc73-410d-b872-de802542933d', 'driver_func-1bd54baa-72f8-46f2-acf2-870f71b5d6ca', 'driver_func-8407c973-4b07-40d8-a12a-76780fa44b1d'} (stimulus_id='handle-worker-cleanup-1782587

In [10]:
cluster

LocalCluster(65c59d57, 'tcp://127.0.0.1:46381', workers=80, threads=80, memory=372.53 GiB)

In [11]:
def suppress_warnings():
    import warnings
    warnings.filterwarnings("ignore", message=".*All-NaN axis encountered*", category=UserWarning)
    warnings.filterwarnings("ignore", category=RuntimeWarning)

client.run(suppress_warnings)

{'tcp://127.0.0.1:32815': None,
 'tcp://127.0.0.1:33195': None,
 'tcp://127.0.0.1:33197': None,
 'tcp://127.0.0.1:33573': None,
 'tcp://127.0.0.1:33817': None,
 'tcp://127.0.0.1:34097': None,
 'tcp://127.0.0.1:34251': None,
 'tcp://127.0.0.1:34379': None,
 'tcp://127.0.0.1:34817': None,
 'tcp://127.0.0.1:35293': None,
 'tcp://127.0.0.1:35563': None,
 'tcp://127.0.0.1:35651': None,
 'tcp://127.0.0.1:35701': None,
 'tcp://127.0.0.1:36015': None,
 'tcp://127.0.0.1:36255': None,
 'tcp://127.0.0.1:36257': None,
 'tcp://127.0.0.1:36509': None,
 'tcp://127.0.0.1:36531': None,
 'tcp://127.0.0.1:36611': None,
 'tcp://127.0.0.1:36737': None,
 'tcp://127.0.0.1:36769': None,
 'tcp://127.0.0.1:36981': None,
 'tcp://127.0.0.1:37235': None,
 'tcp://127.0.0.1:37453': None,
 'tcp://127.0.0.1:37537': None,
 'tcp://127.0.0.1:38491': None,
 'tcp://127.0.0.1:38579': None,
 'tcp://127.0.0.1:38677': None,
 'tcp://127.0.0.1:38755': None,
 'tcp://127.0.0.1:38947': None,
 'tcp://127.0.0.1:39277': None,
 'tcp://

In [29]:
%%time
results = []
start_id = 0
domain = 'd2'
end_id = len(d2_ct_files)
for ii in range(start_id,end_id):
    #print(ii)
    result = dask.delayed(driver_func)(d2_ct_files[ii],d2_dsd_files[ii],d2_cld_files[ii],d2_met_files[ii],csapr_files[0],domain)
    #result = dask.delayed(driver_func)(d3_ct_files[ii],d3_dsd_files[ii],d3_cld_files[ii],d3_met_files[ii],csapr_files[0],domain)
    #result = dask.delayed(driver_func)(d4_ct_files[ii],d4_dsd_files[ii],d4_cld_files[ii],d4_met_files[ii],csapr_files[0],domain)
    results.append(result)

CPU times: user 13.9 ms, sys: 8.67 ms, total: 22.6 ms
Wall time: 16.6 ms


In [30]:
%%time
futures = client.compute(results)  # results is a list of delayed or dask collections

CPU times: user 13.7 ms, sys: 8.31 ms, total: 22 ms
Wall time: 19.5 ms


In [31]:
%%timeprint('Step 1')# Step 1: List .npz filesnpz_files = sorted(glob.glob('/pscratch/sd/m/mckenna/cacti/wrf/temp_cong_dsd_params/LASSO_d2_cong_dsd_params_*.npz'))print('Step 2')# Step 2: Get all keys by inspecting the first filewith np.load(npz_files[0]) as sample:    all_keys = list(sample.keys())print('Step 3')# Step 3: Delayed function to load a single file@delayeddef load_npz_var(filepath, varname):    with np.load(filepath) as data:        return data[varname]print('Step 4')# Step 4: For each variable, collect delayed arrays and concatenatedef extract_and_concat(varname, file_list):    delayed_arrays = [load_npz_var(f, varname) for f in file_list]    return delayed(np.concatenate)(delayed_arrays)print('Step 5')# Step 5: Build delayed concatenation for each variableconcatenated_vars = {    k: extract_and_concat(k, npz_files) for k in all_keys}print('Step 6')# Step 6: Compute in parallelcomputed_values = compute(*concatenated_vars.values())print('Step 7')# Step 7: Reconstruct dictionarywrf_dict = dict(zip(concatenated_vars.keys(), computed_values))print(np.min(wrf_dict['temp']))with open('generated_data/LASSO_d2_dsd_props_cu_only.p', 'wb') as f:    pickle.dump(wrf_dict, f)


Step 1
Step 2
Step 3
Step 4
Step 5
Step 6
Step 7
6.1035156e-05
CPU times: user 10.6 s, sys: 3.66 s, total: 14.2 s
Wall time: 12.9 s
